# Phase 1: Data Preprocessing

This notebook focuses on preparing the explored data for training with `XLM-RoBERTa`.

**Objective**:
- Clean the text (remove URLs, special characters, etc.)
- Tokenize text using `xlm-roberta-base`
- Create PyTorch DataLoaders
- Save processed datasets to disk for the training phase.

In [ ]:
import pandas as pd
import torch
from transformers import XLMRobertaTokenizer
from sklearn.model_selection import train_test_split

MODEL_NAME = 'xlm-roberta-base'
tokenizer = XLMRobertaTokenizer.from_pretrained(MODEL_NAME)

## 1. Text Cleaning Function

In [ ]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    # Remove mentions and hashtags
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = re.sub(r'#[A-Za-z0-9_]+', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Example
sample = "Check out this link: https://example.com #fake @user!!"
print(f"Original: {sample}")
print(f"Cleaned:  {clean_text(sample)}")

## 2. Tokenization and Dataset Creation

In [ ]:
def prepare_data(df, max_len=128):
    """
    Tokenizes the text and returns a PyTorch Dataset-like structure.
    """
    texts = df['text'].apply(clean_text).tolist()
    labels = df['label'].tolist()
    
    encoding = tokenizer.batch_encode_plus(
        texts,
        add_special_tokens=True,
        max_length=max_len,
        return_token_type_ids=False,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )
    
    return {
        'input_ids': encoding['input_ids'],
        'attention_mask': encoding['attention_mask'],
        'labels': torch.tensor(labels, dtype=torch.long)
    }

# dataset_dict = prepare_data(df_train)
